In [10]:
import pandas as pd
import nltk
import re
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

In [ ]:
nltk.download('punkt')

In [2]:
data = pd.read_csv('JEOPARDY_CSV.csv', header=0, skiprows=lambda i: i>0 and random.random() > 0.02)
data.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$800,Edward Teller & this man partnered in 1898 to ...,(Paul) Bonwit
1,4680,2004-12-31,Double Jeopardy!,MUSICAL TRAINS,$800,"During the 1954-1955 Sun sessions, Elvis climb...","the ""Mystery Train"""
2,3751,2000-12-18,Double Jeopardy!,CINEMATIC DICTIONARY,"$1,000",The inventors of this camera-stabilizing devic...,Steadicam
3,3673,2000-07-19,Double Jeopardy!,IN EXILE,$1000,Moshoeshoe II was exiled twice before regainin...,Lesotho
4,5690,2009-05-08,Jeopardy!,MOVIES & TV,$400,"Time magazine said this 2003 Pixar film was ""t...",Finding Nemo


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4371 entries, 0 to 4370
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Show Number  4371 non-null   int64 
 1    Air Date    4371 non-null   object
 2    Round       4371 non-null   object
 3    Category    4371 non-null   object
 4    Value       4293 non-null   object
 5    Question    4371 non-null   object
 6    Answer      4371 non-null   object
dtypes: int64(1), object(6)
memory usage: 239.2+ KB


In [4]:
X = data[[' Question']]
X.head()

,Question
0,Edward Teller & this man partnered in 1898 to ...
1,"During the 1954-1955 Sun sessions, Elvis climb..."
2,The inventors of this camera-stabilizing devic...
3,Moshoeshoe II was exiled twice before regainin...
4,"Time magazine said this 2003 Pixar film was ""t..."


In [5]:
with open('EN-Stopwords.txt', encoding='utf8') as stopwords_file:
    stopwords = stopwords_file.readlines()
stopwords = [line.replace('\n', '') for line in stopwords]
len(stopwords)

1689

In [6]:
dataset = pd.DataFrame(columns=['title_body'])
for index, row in X.iterrows():
    title_body_tokenized = nltk.word_tokenize(row[' Question'])
    title_body_tokenized_filtered = [w.lower() for w in title_body_tokenized if not w.lower() in stopwords]
    s = re.sub('[^\w\s]','', ' '.join(title_body_tokenized_filtered))
    s = re.sub("\d+", "", s)
    dataset.loc[index] = {
        'title_body' : s
    }

In [7]:
dataset

,title_body
0,edward teller partnered sell fashions women
1,sun sessions elvis climbed aboard train six...
2,inventors camerastabilizing device special oscar
3,moshoeshoe exiled regaining southern african c...
4,time magazine pixar film ultimate fishoutofw...
...,...
4366,rustic james a garfield born rustic structures
4367,nuts native australia named scottishborn austr...
4368,capital tourists wellpreserved roman aqueduct...
4369,times writer s scandalot featured maurice gr...


In [9]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(dataset['title_body'])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 28688 stored elements and shape (4371, 11356)>

In [12]:
save_npz('data/X.npz', X)